# 06 – Model Evaluation
Full evaluation of trained models: ROC/PR curves, confusion matrices, calibration, SHAP-style analysis.

In [ ]:
import sys; sys.path.insert(0, '..')
import os, json
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix,
    classification_report, brier_score_loss
)
from sklearn.calibration import calibration_curve
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Load test data and trained models
test_df = pd.read_parquet('../models/test_data.parquet')
y_test = test_df.pop('target')
X_test = test_df

models = {}
for fname in os.listdir('../models'):
    if fname.startswith('model_') and fname.endswith('.pkl'):
        name = fname.replace('model_', '').replace('.pkl', '')
        models[name] = joblib.load(f'../models/{fname}')
print(f'Loaded {len(models)} models: {list(models.keys())}')
print(f'Test set: {X_test.shape} | Class balance: {y_test.value_counts().to_dict()}')

## Metrics Summary

In [ ]:
results = []
for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    report = classification_report(y_test, y_pred, output_dict=True)
    results.append({
        'Model': name,
        'AUC': round(roc_auc_score(y_test, y_prob), 4),
        'AP': round(average_precision_score(y_test, y_prob), 4),
        'F1': round(report['weighted avg']['f1-score'], 4),
        'Precision': round(report['weighted avg']['precision'], 4),
        'Recall': round(report['weighted avg']['recall'], 4),
        'Accuracy': round(report['accuracy'], 4),
        'Brier': round(brier_score_loss(y_test, y_prob), 4),
    })
results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print(results_df.to_string(index=False))

## ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, lw=2, label=f'{name} ({auc:.3f})')
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec, prec, lw=2, label=f'{name} ({ap:.3f})')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set(title='ROC Curves', xlabel='FPR', ylabel='TPR')
axes[0].legend(loc='lower right')
axes[1].set(title='Precision-Recall Curves', xlabel='Recall', ylabel='Precision')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.show()

## Confusion Matrices

In [ ]:
n = len(models)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low','High'], yticklabels=['Low','High'])
    ax.set(title=f'{name}', xlabel='Predicted', ylabel='Actual')
plt.suptitle('Confusion Matrices – Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Calibration Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1],[0,1],'k--', label='Perfect')
for name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker='o', lw=2, label=name)
ax.set(title='Calibration Curves', xlabel='Mean Predicted Probability', ylabel='Fraction of Positives')
ax.legend()
plt.tight_layout()
plt.show()

## Feature Importance Comparison

In [ ]:
tree_models = {k: v for k, v in models.items() if hasattr(v, 'feature_importances_')}
if tree_models:
    n = len(tree_models)
    fig, axes = plt.subplots(1, n, figsize=(8*n, 6))
    if n == 1: axes = [axes]
    for ax, (name, model) in zip(axes, tree_models.items()):
        imp = pd.Series(model.feature_importances_, index=X_test.columns).sort_values(ascending=False).head(15)
        imp.sort_values().plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title(f'Feature Importance – {name}')
    plt.tight_layout()
    plt.show()

## Probability Distribution by Class

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]
y_prob = best_model.predict_proba(X_test)[:, 1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_prob[y_test == 0], bins=50, alpha=0.6, color='steelblue', label='Low Severity (0)', density=True)
ax.hist(y_prob[y_test == 1], bins=50, alpha=0.6, color='tomato', label='High Severity (1)', density=True)
ax.axvline(0.5, color='black', linestyle='--', label='Threshold=0.5')
ax.set(title=f'Predicted Probability Distribution – {best_name}', xlabel='P(High Severity)', ylabel='Density')
ax.legend()
plt.tight_layout()
plt.show()

## Threshold Analysis

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
thresholds = np.arange(0.1, 0.9, 0.05)
f1s, precs, recs = [], [], []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, f1s, 'o-', label='F1', color='green')
ax.plot(thresholds, precs, 's-', label='Precision', color='steelblue')
ax.plot(thresholds, recs, '^-', label='Recall', color='tomato')
ax.axvline(thresholds[np.argmax(f1s)], color='green', linestyle='--', alpha=0.5, label=f'Best F1 threshold={thresholds[np.argmax(f1s)]:.2f}')
ax.set(title=f'Threshold Analysis – {best_name}', xlabel='Threshold', ylabel='Score')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Best F1 threshold: {thresholds[np.argmax(f1s)]:.2f} (F1={max(f1s):.4f})')